# Assignment 1: Prompt-Only Pricing Agent (Baseline)

## Objective
Build a simple LLM price recommender using **prompts only** - no tools, no data lookup, just pure LLM reasoning!

## Requirements
**User Input:**
- Cost Price
- Current Price
- Target Margin (%)
- Competitor Price
- Price Elasticity (Low/Medium/High)

**Agent Action:**
- Computes recommended price using verbal reasoning
- No tools, no data lookup
- Clear explanation of reasoning

## ️Setup & Dependencies

First, let's install the required packages and set up our environment.

In [ ]:
# Install required packages
!pip install -q langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.6 MB/s eta 0:00:00


In [ ]:
# Import required libraries
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
import os
import getpass

In [ ]:
# Set up your Groq API key
print("🔑 Please enter your Groq API key:")
print("(You can get one free at: https://console.groq.com/)")
groq_api_key = getpass.getpass("Groq API Key: ")
os.environ["GROQ_API_KEY"] = groq_api_key
print("✅ API key set successfully!")

🔑 Please enter your Groq API key:
(You can get one free at: https://console.groq.com/)
Groq API Key: ··········
✅ API key set successfully!


## Initialize the Language Model

Let's set up our LLM for pricing recommendations.

In [ ]:
# TODO: Initialize ChatGroq with appropriate parameters
# Hint: llm = ChatGroq() - use model, temperature, max_tokens etc

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=groq_api_key,
    temperature=0.2,
    max_tokens=1000
)

## Prompt Engineering for Pricing

Now let's create effective prompts for our pricing agent. We'll start with a basic approach and then improve it.

### Basic Prompt

**Task:** Create a basic pricing prompt template. Fill in the template below.

In [ ]:
basic_pricing_prompt= PromptTemplate(
    input_variables=["cost_price", "current_price", "target_margin", "competitor_price", "price_elastic"],
    template="""Given the following information:
    Cost Price: ${cost_price}
    Current Price: ${current_price}
    Target Margin: {target_margin}%
    Competitor Price: ${competitor_price}
    Price Elasticity: {price_elasticity}

    Please provide a recommended price and your reasoning."""
)

### Test the Basic Prompt

Let's test our basic prompt with the example from the assignment.

In [ ]:
# Test with the assignment example
test_input = {
    "cost_price": 400,
    "current_price": 599,
    "target_margin": 25,
    "competitor_price": 579,
    "price_elasticity": "Medium"
}

# TODO: Create chain and get response
basic_chain = basic_pricing_prompt | llm
basic_response = basic_chain.invoke(test_input)

print("📊 Basic Pricing Recommendation:")
print(basic_response.content)

📊 Basic Pricing Recommendation:
To determine the recommended price, we need to consider the target margin, competitor price, and price elasticity.

1. **Target Margin**: The target margin is 25%, which means the desired profit margin is 25% of the cost price. 

   Target Price = Cost Price / (1 - Target Margin)
   Target Price = $400 / (1 - 0.25)
   Target Price = $400 / 0.75
   Target Price = $533.33

2. **Competitor Price**: The competitor price is $579. Since the price elasticity is medium, we can adjust the price slightly above the competitor price to maintain a competitive edge.

3. **Price Elasticity**: With a medium price elasticity, the demand for the product is relatively inelastic, meaning that small price changes will not significantly affect demand. However, we still need to be cautious not to price ourselves out of the market.

Considering these factors, a recommended price of $599 is too high, as it's above the target price and the competitor price. A more suitable approa

### Improved Prompt with Chain-of-Thought

**Task:** Create an improved prompt that uses chain-of-thought reasoning for better explanations.

In [ ]:
# TODO: Create an improved prompt with step-by-step reasoning
# Hint: improved_pricing_prompt = PromptTemplate() # input_variables and template.
# Use COT style prompting -
# ANALYSIS STEPS:
# 1. Calculate minimum price based on target margin
# 2. Analyze competitive positioning
# 3. Consider price elasticity impact
# 4. Recommend optimal price

# Chain of Thought prompt
improved_pricing_prompt = PromptTemplate(
    input_variables=["cost_price", "current_price", "target_margin", "competitor_price", "price_elasticity"],
    template="""Given the following information:
    Cost Price: ${cost_price}
    Current Price: ${current_price}
    Target Margin: {target_margin}%
    Competitor Price: ${competitor_price}
    Price Elasticity: {price_elasticity}

    Please follow these steps:
    # 1. Calculate minimum price based on target margin
    # 2. Analyze competitive positioning
    # 3. Consider price elasticity impact
    # 4. Recommend optimal price
    """
)



In [ ]:
# TODO: Test the improved prompt with the same data

test_input = {
    "cost_price": 400,
    "current_price": 599,
    "target_margin": 25,
    "competitor_price": 579,
    "price_elasticity": "Medium"
}

improved_chain = improved_pricing_prompt | llm
improved_response = improved_chain.invoke(test_input)

print("📈 Improved Pricing Analysis:")
print(improved_response.content)

📈 Improved Pricing Analysis:
To determine the optimal price, we'll follow the steps you provided.

**Step 1: Calculate minimum price based on target margin**

To calculate the minimum price based on the target margin, we'll use the following formula:

Minimum Price = Cost Price / (1 - (Target Margin / 100))

Given:
- Cost Price: $400
- Target Margin: 25%

Minimum Price = $400 / (1 - (25 / 100))
Minimum Price = $400 / 0.75
Minimum Price = $533.33

**Step 2: Analyze competitive positioning**

We'll compare our product's price with the competitor's price to determine our competitive positioning.

Given:
- Current Price: $599
- Competitor Price: $579

Since our product's price ($599) is higher than the competitor's price ($579), we're positioned as a premium product. However, we need to consider the price elasticity impact to determine if this positioning is optimal.

**Step 3: Consider price elasticity impact**

Given:
- Price Elasticity: Medium

A medium price elasticity indicates that a

## Class-Based Pricing Agent

Now let's create a reusable class-based pricing agent, similar to the PhysicsTeacherAgent structure.

In [ ]:
class PricingAgent:
    def __init__(self):
        # TODO: Initialize the LLM
        self.llm = llm

        # TODO: Create the system message for pricing agent persona
        self.system_message = SystemMessage(content="""
        You are a seasoned pricing expert. Your goal is to provide data-driven price recommendations
        and clear, concise reasoning based on the input parameters.
        """)

        # TODO: Create the pricing prompt template
        self.pricing_template = PromptTemplate(
            input_variables=["cost_price", "current_price", "target_margin", "competitor_price", "price_elasticity"],
            template="""Given the following information:
    Cost Price: ${cost_price}
    Current Price: ${current_price}
    Target Margin: {target_margin}%
    Competitor Price: ${competitor_price}
    Price Elasticity: {price_elasticity}

    Please follow these steps to provide a detailed pricing recommendation:
    1. Calculate the minimum viable price based on the target margin.
    2. Analyze the current price and competitor's price to understand competitive positioning.
    3. Evaluate the impact of price elasticity on potential demand changes.
    4. Recommend an optimal price, explaining your reasoning considering all factors.
    """
        )

    def get_price_recommendation(self, cost_price, current_price, target_margin, competitor_price, price_elasticity):
        """Get price recommendation with detailed analysis"""
        # TODO: Implement the price recommendation logic
        messages = [
            self.system_message,
            HumanMessage(content=self.pricing_template.format(
                cost_price=cost_price,
                current_price=current_price,
                target_margin=target_margin,
                competitor_price=competitor_price,
                price_elasticity=price_elasticity
            ))
        ]

        response = self.llm.invoke(messages)
        return response.content

    def quick_price_check(self, cost_price, target_margin, competitor_price):
        """Quick price check with minimal inputs"""
        quick_prompt = f"""Quick pricing check:
        Cost: ${cost_price}, Target Margin: {target_margin}%, Competitor: ${competitor_price}

        Provide a quick price recommendation with brief reasoning."""

        response = self.llm.invoke([HumanMessage(content=quick_prompt)])
        return response.content

print("🏗️ Initializing Pricing Agent...")
pricing_agent = PricingAgent()
print("✅ Pricing Agent Ready!")

🏗️ Initializing Pricing Agent...
✅ Pricing Agent Ready!


## Testing Our Pricing Agent

Let's test our pricing agent with various scenarios.

In [ ]:
# Test with the assignment example
print("Testing Assignment Example:")
print("=" * 50)

result = pricing_agent.get_price_recommendation(
    cost_price=400,
    current_price=599,
    target_margin=25,
    competitor_price=579,
    price_elasticity="Medium"
)

print(result)

Testing Assignment Example:
**Step 1: Calculate the Minimum Viable Price (MVP) based on the Target Margin**

To calculate the MVP, we need to determine the minimum price at which the product can be sold to achieve the target margin.

Target Margin = 25% (as a decimal, 0.25)
Cost Price = $400

MVP = (Cost Price / (1 - Target Margin)) 
MVP = ($400 / (1 - 0.25))
MVP = ($400 / 0.75)
MVP = $533.33

**Step 2: Analyze the Current Price and Competitor's Price**

Current Price = $599
Competitor Price = $579

The current price is higher than the MVP ($599 vs $533.33), indicating that the current price is not optimized for the target margin. The competitor's price is also relatively close to the current price, suggesting a competitive market.

**Step 3: Evaluate the Impact of Price Elasticity on Potential Demand Changes**

Price Elasticity = Medium (assuming a medium elasticity coefficient of 0.5)

A medium price elasticity indicates that demand will decrease by approximately 5% for every 1% incr

In [ ]:
# TODO: Test with different scenarios - Add your own test cases!

# Test Case 1: High elasticity scenario
print("\nTest Case 1: High Elasticity Scenario")
print("=" * 50)
result1 = pricing_agent.get_price_recommendation(
    cost_price=100,
    current_price=200,
    target_margin=30,
    competitor_price=180,
    price_elasticity="High"
)
print(result1)

# Test Case 2: Low elasticity scenario
print("\nTest Case 2: Low Elasticity Scenario")
print("=" * 50)
result2 = pricing_agent.get_price_recommendation(
    cost_price=50,
    current_price=100,
    target_margin=40,
    competitor_price=120,
    price_elasticity="Low"
)
print(result2)


Test Case 1: High Elasticity Scenario
**Step 1: Calculate the Minimum Viable Price based on the Target Margin**

To calculate the minimum viable price, we need to determine the price at which the target margin is achieved. The target margin is 30%, which means the desired profit margin is 30% of the cost price.

Cost Price: $100
Target Margin: 30% of $100 = $30

Minimum Viable Price = Cost Price + Target Margin
= $100 + $30
= $130

**Step 2: Analyze the Current Price and Competitor's Price to Understand Competitive Positioning**

Current Price: $200
Competitor Price: $180

The current price is higher than the competitor's price, which indicates a premium pricing strategy. However, the competitor's price is still relatively close to the current price, suggesting a competitive market.

**Step 3: Evaluate the Impact of Price Elasticity on Potential Demand Changes**

Price Elasticity: High

High price elasticity indicates that small changes in price can lead to significant changes in dema

In [ ]:
# Test the quick price check function
print("⚡ Quick Price Check Test:")
print("=" * 30)

quick_result = pricing_agent.quick_price_check(
    cost_price=250,
    target_margin=20,
    competitor_price=350
)
print(quick_result)

⚡ Quick Price Check Test:
To determine a price recommendation, let's calculate the target price based on the target margin and cost.

1. Calculate the target profit: 
   Target Margin = 20% of Cost
   Target Profit = 0.20 * $250 = $50

2. Calculate the target price:
   Target Price = Cost + Target Profit
   Target Price = $250 + $50 = $300

This price recommendation is $50 lower than the competitor's price, which should help you stay competitive while maintaining a 20% margin.


## Experiment with Different Prompting Strategies

Try different prompting approaches and compare the results.

In [ ]:
few_shot_template = PromptTemplate(
    input_variables=["cost_price", "current_price", "target_margin", "competitor_price", "price_elasticity"],
    template="""You are a pricing expert. Here are some examples of good pricing decisions:

Example 1:
Cost: $200, Current: $400, Target Margin: 30%, Competitor: $380, Elasticity: Medium
Recommendation: $375 (maintains margin above 30%, competitive with market, good for medium elasticity)

Example 2:
Cost: $100, Current: $180, Target Margin: 25%, Competitor: $200, Elasticity: Low
Recommendation: $190 (exceeds margin target, leverages low elasticity for higher profit)

Now analyze this case:
Cost: ${cost_price}, Current: ${current_price}, Target Margin: {target_margin}%, Competitor: ${competitor_price}, Elasticity: {price_elasticity}

Recommendation:"""
)

# Test few-shot approach
few_shot_chain = few_shot_template | llm
few_shot_result = few_shot_chain.invoke(test_input)

print("🎯 Few-Shot Prompting Result:")
print(few_shot_result.content)

🎯 Few-Shot Prompting Result:
To determine the optimal price, let's analyze the given information:

- Cost: $400
- Current price: $599
- Target Margin: 25%
- Competitor price: $579
- Elasticity: Medium

First, let's calculate the current margin:

Current Margin = (Current Price - Cost) / Current Price
= ($599 - $400) / $599
= $199 / $599
= 0.332 or 33.2%

Since the current margin (33.2%) is higher than the target margin (25%), we need to adjust the price to meet the target margin. 

To calculate the target price, we'll use the following formula:

Target Price = (Cost * (1 + Target Margin)) / (1 + Current Margin)

Target Price = ($400 * (1 + 0.25)) / (1 + 0.332)
= ($400 * 1.25) / 1.332
= $500 / 1.332
= $375.47

However, considering the competitor's price ($579) and the medium elasticity, we can adjust the target price to be competitive while maintaining the target margin. 

Given the medium elasticity, we can set the target price slightly above the competitor's price to maintain competit

## Compare Different Approaches

Let's compare the different prompting strategies we've implemented.

In [ ]:
# TODO: Create a comparison of all approaches
def compare_pricing_approaches(cost_price, current_price, target_margin, competitor_price, price_elasticity):
    """Compare different prompting approaches for the same input"""

    print(f"📊 PRICING COMPARISON FOR:")
    print(f"Cost: ${cost_price}, Current: ${current_price}, Target Margin: {target_margin}%")
    print(f"Competitor: ${competitor_price}, Elasticity: {price_elasticity}")
    print("="*60)

    # Basic approach
    basic_result = basic_chain.invoke({
        "cost_price": cost_price,
        "current_price": current_price,
        "target_margin": target_margin,
        "competitor_price": competitor_price,
        "price_elasticity": price_elasticity
    })

    # Improved approach
    improved_result = improved_chain.invoke({
        "cost_price": cost_price,
        "current_price": current_price,
        "target_margin": target_margin,
        "competitor_price": competitor_price,
        "price_elasticity": price_elasticity
    })

    # Class-based approach
    class_result = pricing_agent.get_price_recommendation(
        cost_price, current_price, target_margin, competitor_price, price_elasticity
    )

    print("🔴 BASIC APPROACH:")
    print(basic_result.content[:200] + "...\n")

    print("🟡 IMPROVED APPROACH:")
    print(improved_result.content[:200] + "...\n")

    print("🟢 CLASS-BASED APPROACH:")
    print(class_result[:200] + "...")

# Run comparison
compare_pricing_approaches(400, 599, 25, 579, "Medium")

📊 PRICING COMPARISON FOR:
Cost: $400, Current: $599, Target Margin: 25%
Competitor: $579, Elasticity: Medium
🔴 BASIC APPROACH:
To determine a recommended price, we'll consider the target margin and the competitor's price. 

First, let's calculate the target selling price based on the target margin. The target margin is 25%, s...

🟡 IMPROVED APPROACH:
To determine the optimal price, we'll follow the steps you provided.

**Step 1: Calculate minimum price based on target margin**

To calculate the minimum price based on the target margin, we'll use t...

🟢 CLASS-BASED APPROACH:
**Step 1: Calculate the Minimum Viable Price based on the Target Margin**

To calculate the minimum viable price, we need to determine the target revenue based on the target margin and then subtract t...
